In [12]:
# ── Data loading ─────────────────────────────────────────────────────────────
# Loads keypoints, appearance gallery, shape betas, and pose sequences
# for every person in every camera.

import numpy as np
from pathlib import Path
from itertools import combinations
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
from scipy.spatial.transform import Rotation as Rot

SCENE = 'Gym_011_burpee2'
BASE  = Path(f'/iopsstor/scratch/cscs/tnanni/ghost_outputs/rich_test/{SCENE}')

def load_tracks(cam: str) -> dict:
    body_dir = BASE / cam / 'body_data'
    persons = {}

    # Load appearance gallery (DINOv3 features)
    app_gallery: dict[int, tuple] = {}
    gallery_path = body_dir / 'appearance_gallery.npz'
    if gallery_path.exists():
        gdata = np.load(str(gallery_path))
        for k in gdata.files:
            if k.endswith('_conf'):
                continue
            pid_key  = int(k)
            conf_key = f'{k}_conf'
            feats    = gdata[k]
            confs    = gdata[conf_key] if conf_key in gdata.files else np.ones(len(feats), dtype=np.float32)
            app_gallery[pid_key] = (feats, confs)

    for npz_path in sorted(body_dir.glob('person_*.npz')):
        pid = int(npz_path.stem.split('_')[1])
        with np.load(str(npz_path)) as d:
            if 'pred_keypoints_3d' not in d or 'frame_indices' not in d:
                continue
            entry = {
                'kpts3d': d['pred_keypoints_3d'].copy(),  # (T, 70, 3) camera-local root-centred
                'frames': d['frame_indices'].copy(),
            }
            # Camera-frame root position
            for key in ('pred_cam_t', 'smplx_transl'):
                if key in d:
                    entry['pred_cam_t'] = d[key].copy()
                    break

            # Shape: confidence-weighted median of betas
            if 'smplx_betas' in d:
                betas = d['smplx_betas'].astype(np.float32)
                if len(betas) > 0:
                    conf = d.get('pred_joint_confidence')
                    if conf is not None and len(conf) == len(betas):
                        frame_conf = np.mean(conf, axis=-1).astype(np.float32)
                        total_w = frame_conf.sum()
                        shape_med = ((frame_conf[:, None] * betas).sum(0) / total_w
                                     if total_w > 0 else np.median(betas, axis=0).astype(np.float32))
                    else:
                        shape_med = np.median(betas, axis=0).astype(np.float32)
                    norm = np.linalg.norm(shape_med)
                    entry['shape_feat'] = shape_med / norm if norm > 0 else shape_med

            # Pose: per-frame canonical joint vectors (root-relative, global-orient removed)
            if 'smplx_global_orient' in d:
                kps = d['pred_keypoints_3d'].astype(np.float32)
                gorient = d['smplx_global_orient'].astype(np.float32)
                N_kps = len(kps)
                if N_kps > 0:
                    kps_rel = kps - kps[:, 0:1, :]
                    rot_mats = Rot.from_rotvec(gorient).inv().as_matrix()
                    kps_canon = np.einsum('nij,nkj->nki', rot_mats, kps_rel).astype(np.float32)
                    pose_vecs = kps_canon.reshape(N_kps, -1)
                    norms_p   = np.linalg.norm(pose_vecs, axis=1, keepdims=True)
                    pose_vecs = np.where(norms_p > 1e-6, pose_vecs / norms_p, pose_vecs)
                    conf_kps  = d.get('pred_joint_confidence')
                    pose_confs = (np.mean(conf_kps, axis=-1).astype(np.float32)
                                  if conf_kps is not None and len(conf_kps) == N_kps
                                  else np.ones(N_kps, dtype=np.float32))
                    entry['pose_vecs']  = pose_vecs
                    entry['pose_confs'] = pose_confs

        entry['app_feat'] = app_gallery.get(pid)  # (feats, confs) or None
        persons[pid] = entry
    return persons

all_tracks = {}
for cam_dir in sorted(BASE.iterdir()):
    if not (cam_dir / 'body_data').exists():
        continue
    tracks = load_tracks(cam_dir.name)
    if tracks:
        all_tracks[cam_dir.name] = tracks

cam_list = sorted(all_tracks.keys())
print(f'Loaded {len(all_tracks)} cameras:')
for cam, persons in sorted(all_tracks.items()):
    print(f'  {cam}:')
    for pid, d in sorted(persons.items()):
        n   = len(d['frames'])
        std = float(d['kpts3d'].std(axis=0).mean()) if n > 0 else 0.0
        has_app   = d['app_feat'] is not None
        has_shape = 'shape_feat' in d
        has_pose  = 'pose_vecs' in d
        print(f'    P{pid}: {n} frames  joint_std={std:.4f} m  '
              f'app={has_app}  shape={has_shape}  pose={has_pose}')

Loaded 4 cameras:
  cam_02:
    P1: 479 frames  joint_std=0.2971 m  app=True  shape=True  pose=True
    P2: 479 frames  joint_std=0.0204 m  app=True  shape=True  pose=True
  cam_03:
    P1: 479 frames  joint_std=0.2845 m  app=True  shape=True  pose=True
  cam_04:
    P1: 479 frames  joint_std=0.2887 m  app=True  shape=True  pose=True
    P2: 479 frames  joint_std=0.0185 m  app=True  shape=True  pose=True
  cam_05:
    P1: 478 frames  joint_std=0.2883 m  app=True  shape=True  pose=True
    P2: 444 frames  joint_std=0.0502 m  app=True  shape=True  pose=True
    P3: 478 frames  joint_std=0.0494 m  app=True  shape=True  pose=True
    P4: 478 frames  joint_std=0.0467 m  app=True  shape=True  pose=True
    P5: 478 frames  joint_std=0.0126 m  app=True  shape=True  pose=True
    P6: 478 frames  joint_std=0.0086 m  app=True  shape=True  pose=True
    P7: 276 frames  joint_std=0.0935 m  app=True  shape=True  pose=True


In [13]:
# ── Core functions ────────────────────────────────────────────────────────────

# ── Thresholds ────────────────────────────────────────────────────────────────
# Appearance
HIGH_APP_THR  = 0.60  # accept on appearance alone (anchor candidate)
LOW_APP_THR   = 0.35  # reject on appearance alone
APP_W         = 0.50  # appearance weight in sim matrix
SHAPE_W       = 0.20  # shape weight
POSE_W        = 0.30  # pose xcorr weight

# Geometry
MIN_OVERLAP          = 30    # min common frames for affine fit
MIN_ANCHOR_STD       = 0.05  # m — anchor must have temporal motion
MIN_A_SINGULAR_VALUE = 0.3   # reject degenerate affines
GEO_MATCH_THR        = 0.25  # m — geometric acceptance threshold


# ── Geometric helpers (from reid_geometric.ipynb) ─────────────────────────────

def affine_fit(src: np.ndarray, dst: np.ndarray):
    """12-DOF affine via least squares. src, dst: (N, 3)."""
    X = np.concatenate([src, np.ones((len(src), 1), dtype=src.dtype)], axis=1)
    valid = np.isfinite(X).all(axis=1) & np.isfinite(dst).all(axis=1)
    if valid.sum() < 12:
        return np.eye(3), np.zeros(3)
    M, _, _, _ = np.linalg.lstsq(X[valid], dst[valid], rcond=None)
    return M[:3].T, M[3]

def apply_T(A, t_vec, kpts):
    shape = kpts.shape
    return (A @ kpts.reshape(-1, 3).T).T.reshape(shape) + t_vec

def get_aligned_slices(data_b, data_a, delta=0):
    fb_to_idx = {int(f): i for i, f in enumerate(data_b['frames'])}
    fa_to_idx = {int(f): i for i, f in enumerate(data_a['frames'])}
    common = sorted(f for f in fb_to_idx if f + delta in fa_to_idx)
    if len(common) < MIN_OVERLAP:
        return None, None
    idx_b = [fb_to_idx[f]         for f in common]
    idx_a = [fa_to_idx[f + delta] for f in common]
    return data_b['kpts3d'][idx_b], data_a['kpts3d'][idx_a]

def geo_rmse(A, t_vec, data_b, data_a, delta=0):
    src, dst = get_aligned_slices(data_b, data_a, delta)
    if src is None:
        return float('inf')
    pred   = apply_T(A, t_vec, src)
    result = float(np.sqrt(((pred - dst) ** 2).sum(-1).mean()))
    return result if np.isfinite(result) else float('inf')


# ── Appearance helpers (from cross_view_reidentifier.py) ─────────────────────

def chamfer_sim(feats_a, confs_a, feats_b, confs_b):
    S = feats_a @ feats_b.T
    return 0.5 * (float(S.max(axis=1).mean()) + float(S.max(axis=0).mean()))

def xcorr_sim(feats_a, feats_b, min_overlap=30):
    fa = feats_a - feats_a.mean(axis=0)
    fb = feats_b - feats_b.mean(axis=0)
    na = np.linalg.norm(fa, axis=1, keepdims=True)
    nb = np.linalg.norm(fb, axis=1, keepdims=True)
    fa = np.where(na > 1e-6, fa / na, fa)
    fb = np.where(nb > 1e-6, fb / nb, fb)
    S  = fa @ fb.T
    best_score = -1.0
    for tau in range(-(len(feats_b) - 1), len(feats_a)):
        diag = np.diagonal(S, offset=-tau)
        if len(diag) < min_overlap:
            continue
        score = float(diag.mean())
        if score > best_score:
            best_score = score
    return float(max(0.0, best_score))

def appearance_sim_matrix(pids_a, pids_b, tracks_a, tracks_b):
    Na, Nb = len(pids_a), len(pids_b)
    sim_mat    = np.zeros((Na, Nb), dtype=np.float32)
    weight_mat = np.zeros((Na, Nb), dtype=np.float32)

    # Appearance
    for i, pa in enumerate(pids_a):
        app_a = tracks_a[pa].get('app_feat')
        if app_a is None:
            continue
        feats_a, confs_a = app_a
        for j, pb in enumerate(pids_b):
            app_b = tracks_b[pb].get('app_feat')
            if app_b is None:
                continue
            feats_b, confs_b = app_b
            s = chamfer_sim(feats_a, confs_a, feats_b, confs_b)
            sim_mat[i, j]    += APP_W * s
            weight_mat[i, j] += APP_W

    # Shape
    shape_a = [tracks_a[p].get('shape_feat') for p in pids_a]
    shape_b = [tracks_b[p].get('shape_feat') for p in pids_b]
    mask_a  = np.array([f is not None for f in shape_a], dtype=np.float32)
    mask_b  = np.array([f is not None for f in shape_b], dtype=np.float32)
    if mask_a.any() and mask_b.any():
        zero  = np.zeros_like(next(f for f in shape_a if f is not None))
        mat_a = np.stack([f if f is not None else zero for f in shape_a])
        mat_b = np.stack([f if f is not None else zero for f in shape_b])
        shape_sim = mat_a @ mat_b.T
        shape_w   = np.outer(mask_a, mask_b) * SHAPE_W
        sim_mat    += shape_w * shape_sim
        weight_mat += shape_w

    # Pose xcorr
    for i, pa in enumerate(pids_a):
        pv_a = tracks_a[pa].get('pose_vecs')
        if pv_a is None:
            continue
        for j, pb in enumerate(pids_b):
            pv_b = tracks_b[pb].get('pose_vecs')
            if pv_b is None:
                continue
            s = xcorr_sim(pv_a, pv_b)
            sim_mat[i, j]    += POSE_W * s
            weight_mat[i, j] += POSE_W

    return np.where(weight_mat > 0, sim_mat / weight_mat, 0.0)


# ── Union-Find ────────────────────────────────────────────────────────────────

class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[ry] = rx

def cluster_cams(uf, node):
    root = uf.find(node)
    return {cam for cam, pid in uf.parent if uf.find((cam, pid)) == root}

print('Functions loaded.')

Functions loaded.


In [14]:
# ── Combined appearance + geometry ReID ───────────────────────────────────────
#
# Per camera pair:
#   1. Appearance sim matrix (appearance + shape + pose) → Hungarian assignment
#   2. High-confidence pairs (sim >= HIGH_APP_THR) with temporal motion
#      → fit inter-camera 12-DOF affine from their keypoints (anchors)
#   3. Uncertain pairs (LOW_APP_THR <= sim < HIGH_APP_THR)
#      → accept if geometric RMSE < GEO_MATCH_THR under the fitted affine
#   4. Union-Find merge with same-camera conflict guard

uf = UnionFind()
for cam in cam_list:
    for pid in all_tracks[cam]:
        uf.find((cam, pid))

for cam_a, cam_b in combinations(cam_list, 2):
    tracks_a = all_tracks[cam_a]
    tracks_b = all_tracks[cam_b]
    pids_a   = sorted(tracks_a.keys())
    pids_b   = sorted(tracks_b.keys())

    # ── Step 1: appearance-based Hungarian ───────────────────────────────────
    sim_mat  = appearance_sim_matrix(pids_a, pids_b, tracks_a, tracks_b)
    row_ind, col_ind = linear_sum_assignment(1.0 - sim_mat)
    assignment = [(pids_a[r], pids_b[c], float(sim_mat[r, c]))
                  for r, c in zip(row_ind, col_ind)]

    print(f'\n{cam_a} × {cam_b}:')
    for pa, pb, s in assignment:
        print(f'  app: P{pa} ↔ P{pb}  sim={s:.3f}')

    # ── Step 2: anchor extraction and affine estimation ───────────────────────
    # Anchors: high appearance confidence AND at least one dynamic track
    anchors = [
        (pa, pb) for pa, pb, s in assignment
        if s >= HIGH_APP_THR
        and (float(tracks_a[pa]['kpts3d'].std(axis=0).mean()) >= MIN_ANCHOR_STD
             or float(tracks_b[pb]['kpts3d'].std(axis=0).mean()) >= MIN_ANCHOR_STD)
    ]

    A, t_vec = None, None
    if anchors:
        all_src, all_dst = [], []
        for pa, pb in anchors:
            src, dst = get_aligned_slices(tracks_b[pb], tracks_a[pa])
            if src is not None:
                all_src.append(src.reshape(-1, 3))
                all_dst.append(dst.reshape(-1, 3))
        if all_src:
            A_cand, t_cand = affine_fit(np.concatenate(all_src), np.concatenate(all_dst))
            sv = np.linalg.svd(A_cand, compute_uv=False)
            if sv.min() >= MIN_A_SINGULAR_VALUE:
                A, t_vec = A_cand, t_cand
                print(f'  affine from {len(anchors)} anchor(s)  sv=[{sv.min():.2f},{sv.max():.2f}]')
            else:
                print(f'  affine degenerate (sv_min={sv.min():.2f}) — geometry skipped')
    else:
        print(f'  no anchors (no high-confidence + dynamic pair) — geometry skipped')

    # ── Steps 3-4: classify and merge ────────────────────────────────────────
    for pa, pb, sim in assignment:
        node_a = (cam_a, pa)
        node_b = (cam_b, pb)

        if sim >= HIGH_APP_THR:
            decision = 'high-conf appearance'
            accept   = True

        elif sim < LOW_APP_THR:
            decision = 'low-conf appearance'
            accept   = False

        else:  # uncertain band — use geometry
            if A is None:
                decision = 'uncertain, no affine → rejected'
                accept   = False
            else:
                rmse   = geo_rmse(A, t_vec, tracks_b[pb], tracks_a[pa])
                accept = rmse < GEO_MATCH_THR
                decision = f'geo RMSE={rmse:.3f} m → {"accepted" if accept else "rejected"}'

        if accept:
            conflict = cluster_cams(uf, node_a) & cluster_cams(uf, node_b)
            if conflict:
                print(f'  P{pa} ↔ P{pb}  sim={sim:.3f}  [{decision}]  '
                      f'[conflict {conflict}]')
            else:
                uf.union(node_a, node_b)
                print(f'  P{pa} ↔ P{pb}  sim={sim:.3f}  [{decision}]  [merged]')
        else:
            print(f'  P{pa} ↔ P{pb}  sim={sim:.3f}  [{decision}]  [skipped]')

# ── Summary ───────────────────────────────────────────────────────────────────
by_gid = defaultdict(list)
for cam in cam_list:
    for pid in all_tracks[cam]:
        by_gid[uf.find((cam, pid))].append((cam, pid))

print('\n=== Final Clusters ===')
for gid, (_, members) in enumerate(sorted(by_gid.items()), 1):
    print(f'  Person {gid}: {"  ".join(f"{c}/P{p}" for c, p in sorted(members))}')


cam_02 × cam_03:
  app: P1 ↔ P1  sim=0.849
  affine from 1 anchor(s)  sv=[0.95,1.02]
  P1 ↔ P1  sim=0.849  [high-conf appearance]  [merged]

cam_02 × cam_04:
  app: P1 ↔ P1  sim=0.964
  app: P2 ↔ P2  sim=0.843
  affine from 1 anchor(s)  sv=[0.97,1.05]
  P1 ↔ P1  sim=0.964  [high-conf appearance]  [merged]
  P2 ↔ P2  sim=0.843  [high-conf appearance]  [merged]

cam_02 × cam_05:
  app: P1 ↔ P1  sim=0.915
  app: P2 ↔ P4  sim=0.669
  affine from 1 anchor(s)  sv=[0.99,1.06]
  P1 ↔ P1  sim=0.915  [high-conf appearance]  [merged]
  P2 ↔ P4  sim=0.669  [high-conf appearance]  [merged]

cam_03 × cam_04:
  app: P1 ↔ P1  sim=0.850
  affine from 1 anchor(s)  sv=[0.98,1.01]
  P1 ↔ P1  sim=0.850  [high-conf appearance]  [conflict {'cam_04', 'cam_02', 'cam_03', 'cam_05'}]

cam_03 × cam_05:
  app: P1 ↔ P1  sim=0.788
  affine from 1 anchor(s)  sv=[0.99,1.06]
  P1 ↔ P1  sim=0.788  [high-conf appearance]  [conflict {'cam_04', 'cam_02', 'cam_03', 'cam_05'}]

cam_04 × cam_05:
  app: P1 ↔ P1  sim=0.913
  a